In [1]:
!pip install -q dspy-ai

# DSPy Crash Course — Part 2: Multi-Model Setup & Modules

> **Reference:** https://dspy.ai/learn/programming/modules/  
> **What you'll learn:**
> - Setting up and switching between multiple LMs
> - All built-in DSPy modules (`Predict`, `ChainOfThought`, `ReAct`, etc.)
> - Composing modules into custom programs with `dspy.Module`
> - Tracking LM usage (tokens + cost)

---

## Table of Contents
1. [Setup — Multiple LMs (Groq 20b / 120b)](#1-setup)
2. [Switching Models — `dspy.configure` vs `dspy.context`](#2-switching-models)
3. [Built-in Modules Overview](#3-built-in-modules)
4. [Predict — The Basic Module](#4-predict)
5. [ChainOfThought — Reasoning Before Answering](#5-chainofthought)
6. [ProgramOfThought — Code as Reasoning](#6-programofthought)
7. [ReAct — Agent with Tools](#7-react)
8. [Composing Modules into a Custom Program](#8-custom-program)
9. [Tracking LM Usage (Tokens & Cost)](#9-usage-tracking)
10. [Comparing Models Side by Side](#10-model-comparison)

In [2]:
import dspy
from dspy import LM, Predict, Signature, settings, context

from google.colab import userdata

# ── Groq credentials ───────────────────────────────────────────────
groq_api_key = userdata.get('GROQ_API')
api_base     = 'https://api.groq.com/openai/v1'

# Two Groq-hosted OpenAI-compatible models
MODEL_20B  = "groq/openai/gpt-oss-20b"
MODEL_120B = "groq/openai/gpt-oss-120b"

print("DSPy version:", dspy.__version__)
print("API key loaded:", "✓" if groq_api_key else "✗ check Colab secrets")

DSPy version: 3.1.3
API key loaded: ✓


In [3]:
# Create two separate LM objects — each can be used independently or swapped in globally
gpt_20b = LM(
    model=MODEL_20B,
    api_base=api_base,
    api_key=groq_api_key,
)

gpt_120b = LM(
    model=MODEL_120B,
    api_base=api_base,
    api_key=groq_api_key,
)

# Set gpt_20b as the global default
settings.configure(lm=gpt_20b)
print("Default LM:", gpt_20b.model)

Default LM: groq/openai/gpt-oss-20b


---
## 2. Switching Models

DSPy offers two ways to control which LM is used:

| Method | Scope | Use When |
|---|---|---|
| `dspy.configure(lm=...)` | Global | Changing default for all subsequent calls |
| `with dspy.context(lm=...)` | Block-scoped | Temporarily using a different model, thread-safe |

> **Thread-safe:** Both methods are safe to use in multi-threaded/async code.

In [4]:
qa = dspy.Predict("question -> answer")
question = "Who is the world's fastest man and woman?"

# --- Using global default (gpt_20b) ---
response_20b = qa(question=question)
print(f"[20b]  {response_20b.answer}\n")

# --- Temporarily override with gpt_120b using context manager ---
with dspy.context(lm=gpt_120b):
    response_120b = qa(question=question)
    print(f"[120b] {response_120b.answer}\n")

# --- Switch global default permanently ---
dspy.configure(lm=gpt_120b)
response_new_default = qa(question=question)
print(f"[new default=120b] {response_new_default.answer}")

# Restore 20b as default for the rest of the notebook
dspy.configure(lm=gpt_20b)

[20b]  Usain Bolt is the world’s fastest man (100‑m world record of 9.58 s), and Florence Griffith‑Joyner is the world’s fastest woman (100‑m world record of 10.49 s).

[120b] Usain Bolt is the world’s fastest man (he holds the men’s 100 m world record of 9.58 seconds, set in 2009), and Florence Griffith‑Joyner is the world’s fastest woman (she holds the women’s 100 m world record of 10.49 seconds, set in 1988).

[new default=120b] Usain Bolt is the world’s fastest man (he holds the men’s 100 m world record of 9.58 seconds, set in 2009), and Florence Griffith‑Joyner is the world’s fastest woman (she holds the women’s 100 m world record of 10.49 seconds, set in 1988).


---
## 3. Built-in Modules Overview

All DSPy modules share the same pattern:  
**declare with a signature → call with inputs → access outputs**

| Module | What it does | When to use |
|---|---|---|
| `dspy.Predict` | Direct LM call | Simple tasks, baseline |
| `dspy.ChainOfThought` | Adds `reasoning` before answer | Improves quality on reasoning tasks |
| `dspy.ProgramOfThought` | Outputs + executes code | Math, data processing |
| `dspy.ReAct` | Agent with tools (search, code, etc.) | Multi-step tasks needing external info |
| `dspy.MultiChainComparison` | Runs N chains, picks best | When you need high-confidence answers |
| `dspy.majority` | Votes across predictions | Ensemble / self-consistency |

---
## 4. Predict — The Basic Module

`dspy.Predict` is the foundation of all other modules. It takes a signature and calls the LM once.

In [5]:
# Pattern: declare → call → access output by field name
pred = dspy.Predict("question -> answer")
result = pred(question="What is machine learning?")

print("Answer:", result.answer)
print()

# Different output field names = semantically different prompts
pred_summary = dspy.Predict("question -> summary")
result2 = pred_summary(question="What capabilities does a large language model have?")
print("Summary:", result2.summary)

Answer: Machine learning is a branch of artificial intelligence that focuses on developing algorithms and statistical models that enable computers to learn from and make predictions or decisions based on data, rather than relying solely on explicit, hand‑coded instructions. By identifying patterns, relationships, and insights in large datasets, machine learning systems improve their performance over time through experience and feedback.

Summary: Large language models can understand and generate natural language text with high fluency, enabling a wide range of applications such as machine translation, summarization, question answering, dialogue systems, creative writing, content moderation, code generation, and sentiment or intent analysis. They excel at zero‑shot and few‑shot learning, allowing them to adapt quickly to new tasks or domains with minimal additional training. They can also perform classification, information extraction, and simple reasoning tasks by leveraging patterns l

---
## 5. ChainOfThought — Reasoning Before Answering

`dspy.ChainOfThought` is a drop-in replacement for `Predict` that instructs the LM to reason step-by-step before committing to the answer.  
It automatically adds a `reasoning` field to the output.

> **Tip:** Simply swapping `Predict` → `ChainOfThought` often improves quality with no other changes.

In [6]:
cot = dspy.ChainOfThought("question -> answer")
result = cot(question="Two dice are tossed. What is the probability that the sum equals two?")

print("Reasoning:", result.reasoning)
print()
print("Answer:", result.answer)
print()

# ChainOfThought with a document summarization task
summarize_cot = dspy.ChainOfThought("document -> summary")
doc = """NASA's Apollo 11 mission in July 1969 was the first crewed lunar landing.
Commander Neil Armstrong and Lunar Module Pilot Buzz Aldrin landed the lunar module Eagle
in the Sea of Tranquility. Armstrong became the first human to step onto the lunar surface,
followed by Aldrin."""
r = summarize_cot(document=doc)
print("Summary:", r.summary)
print("Reasoning:", r.reasoning)

Reasoning: To find the probability that the sum of two tossed dice equals two, we count the favorable outcomes and divide by the total number of possible outcomes.

- Each die has 6 faces, so two dice produce \(6 \times 6 = 36\) equally likely ordered pairs.
- The sum equals 2 only when both dice show a 1: the single outcome (1, 1).

Thus the number of favorable outcomes is 1, and the probability is:
\[
P(\text{sum}=2) = \frac{1}{36}.
\]

Answer: \(\displaystyle \frac{1}{36}\)

Summary: NASA's Apollo 11 mission (July 1969) marked the first crewed lunar landing. Commander Neil Armstrong and Lunar Module Pilot Buzz Aldrin landed the Eagle in the Sea of Tranquility, with Armstrong becoming the first human to walk on the moon, followed by Aldrin.
Reasoning: The user provided a short historical description of Apollo 11. The task is to produce a concise summary of that information, following the specified template. The summary should capture the essential facts: the mission, date, significan

---
## 6. ProgramOfThought — Code as Reasoning

`dspy.ProgramOfThought` makes the LM write Python code to solve the problem. The code is executed, and the result becomes the answer.  
Best for: math, counting, data processing tasks where code is more reliable than prose.

In [7]:
cot = dspy.ChainOfThought("question -> answer: float")

problems = [
    "What is 15% of 240?",
    "A train travels 120 km in 1.5 hours. What is its average speed in km/h?",
    "What is the area of a circle with radius 7? (use pi=3.14159)",
]

for p in problems:
    r = cot(question=p)
    print(f"Q: {p}")
    print(f"A: {r.answer}\n")

Q: What is 15% of 240?
A: 36.0

Q: A train travels 120 km in 1.5 hours. What is its average speed in km/h?
A: 80.0

Q: What is the area of a circle with radius 7? (use pi=3.14159)
A: 153.93791



---
## 7. ReAct — Agent with Tools

`dspy.ReAct` is an **agent** that can interleave reasoning and tool calls.  
You provide Python functions as tools; DSPy converts them into tool descriptions automatically.

```python
dspy.ReAct(signature, tools=[fn1, fn2, ...], max_iters=5)
```

In [8]:
def calculator(expression: str) -> str:
    """Evaluate a Python math expression and return the result as a string.
    Examples: '2 + 2', '15 * 0.15', 'round(3.14159 * 7**2, 4)'"""
    try:
        result = eval(expression, {"__builtins__": {}}, {"round": round, "abs": abs})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

def get_current_year() -> str:
    """Return the current year."""
    return "2026"

# Build the ReAct agent
agent = dspy.ReAct("question -> answer", tools=[calculator, get_current_year], max_iters=4)

result = agent(question="How many years ago did the Apollo 11 moon landing happen (it was in 1969)?")
print("Answer:", result.answer)

Answer: 57


---
## 8. Composing Modules into a Custom Program

A **`dspy.Module`** is like a `torch.nn.Module` for LM programs.  
- Subclass `dspy.Module`  
- Declare sub-modules in `__init__`  
- Implement the logic in `forward`

This makes your program **optimizable** — DSPy can tune prompts for the whole pipeline.

In [9]:
# --- Example: A two-step RAG-style pipeline ---

class SimpleQA(dspy.Module):
    """A minimal two-step program:
    1. Generate a search-friendly query from the question
    2. Answer the question using that query + context
    """

    def __init__(self):
        # Declare sub-modules — each can be independently optimized
        self.generate_query  = dspy.ChainOfThought("question -> search_query")
        self.answer_question = dspy.ChainOfThought("question, context -> answer")

    def forward(self, question: str, context: str = "") -> dspy.Prediction:
        # Step 1: Reformulate question into a search query
        query_result = self.generate_query(question=question)
        search_query = query_result.search_query

        # Step 2: Answer using the reformulated query and any context
        answer_result = self.answer_question(
            question=search_query, context=context or question
        )
        return dspy.Prediction(
            search_query=search_query,
            answer=answer_result.answer,
            reasoning=answer_result.reasoning,
        )

# Instantiate and run
pipeline = SimpleQA()
result = pipeline(
    question="What makes transformer architecture different from RNNs?",
    context="Transformers use self-attention; RNNs process tokens sequentially."
)

print("Search Query:", result.search_query)
print()
print("Reasoning:", result.reasoning)
print()
print("Answer:", result.answer)

Search Query: transformer architecture differences from RNNs key concepts comparison

Reasoning: Transformers and RNNs (including LSTMs/GRUs) are two fundamentally different neural network architectures designed for sequence modeling.

**RNNs**

* **Processing style** – Sequential. Each token is fed one after another, and the hidden state is updated step‑by‑step.
* **Information flow** – Through a single hidden state vector that carries information from all previous tokens.
* **Memory limitation** – Long‑range dependencies suffer from vanishing or exploding gradients; gating mechanisms (LSTM, GRU) mitigate this but still struggle with very long contexts.
* **Parallelism** – Limited. Tokens must be processed in order, which slows training and inference on modern GPUs/TPUs.
* **Typical use‑cases** – Early language models, speech, time‑series where sequence order is crucial.

**Transformers**

* **Processing style** – Parallel. All tokens are processed at once via self‑attention.
* **Info

---
## 9. Tracking LM Usage (Tokens & Cost)

Enable `track_usage=True` in `dspy.configure` to get per-call token counts and cost.  
Access them via `prediction.get_lm_usage()` — returns a dict keyed by model name.

> **Note:** Cached responses return `{}` (no usage counted for cache hits).

In [10]:
# Enable usage tracking globally
dspy.configure(lm=gpt_20b, track_usage=True)

class UsageDemo(dspy.Module):
    def __init__(self):
        self.step1 = dspy.ChainOfThought("question -> answer")
        self.step2 = dspy.ChainOfThought("question, answer -> confidence: float")

    def forward(self, question):
        ans    = self.step1(question=question)
        conf   = self.step2(question=question, answer=ans.answer)
        return dspy.Prediction(answer=ans.answer, confidence=conf.confidence)

program = UsageDemo()
out = program(question="What is the speed of light?")

print("Answer:", out.answer)
print("Confidence:", out.confidence)
print()
print("LM Usage:")
import json
usage = out.get_lm_usage()
print(json.dumps(usage, indent=2, default=str))

Answer: 299,792,458 meters per second.
Confidence: 0.99

LM Usage:
{
  "groq/openai/gpt-oss-20b": {
    "completion_tokens": 432,
    "prompt_tokens": 525,
    "total_tokens": 957,
    "completion_tokens_details": {
      "accepted_prediction_tokens": null,
      "audio_tokens": null,
      "reasoning_tokens": 160,
      "rejected_prediction_tokens": null,
      "text_tokens": null,
      "image_tokens": null
    },
    "prompt_tokens_details": null,
    "queue_time": 0.006718512,
    "prompt_time": 0.026891248,
    "completion_time": 0.443097001,
    "total_time": 0.469988249
  }
}


---
## 10. Comparing Models Side by Side

A practical pattern: run the same pipeline with different models and compare outputs.

In [11]:
questions = [
    "Explain the difference between supervised and unsupervised learning in one sentence.",
    "What is a transformer in deep learning?",
    "What is the capital of Australia?",
]

cot_pred = dspy.ChainOfThought("question -> answer")

print("=" * 70)
for q in questions:
    print(f"\nQ: {q}")

    with dspy.context(lm=gpt_20b):
        r_20b = cot_pred(question=q)
    print(f"  [20b]  {r_20b.answer}")

    with dspy.context(lm=gpt_120b):
        r_120b = cot_pred(question=q)
    print(f"  [120b] {r_120b.answer}")

print("=" * 70)


Q: Explain the difference between supervised and unsupervised learning in one sentence.
  [20b]  Supervised learning uses labeled data to learn a mapping from inputs to outputs, while unsupervised learning discovers hidden patterns in unlabeled data.
  [120b] Supervised learning uses labeled data to learn a mapping from inputs to outputs, whereas unsupervised learning works with unlabeled data to discover hidden patterns or structure.

Q: What is a transformer in deep learning?
  [20b]  A transformer is a deep learning model architecture that relies entirely on self‑attention mechanisms to process sequential data. Unlike recurrent or convolutional networks, transformers allow each token in a sequence to directly attend to every other token, enabling efficient learning of long‑range dependencies. The basic transformer consists of stacked encoder and decoder layers, each containing multi‑head self‑attention, feed‑forward networks, residual connections, and layer normalization. This desi

---
## Quick Reference — Multi-Model Cheatsheet

```python
import dspy

# ── Multiple LMs ──────────────────────────────────────────────────
lm_small  = dspy.LM("groq/llama-3.1-8b-instant",  api_key="...")
lm_large  = dspy.LM("groq/llama-3.3-70b-versatile", api_key="...")

dspy.configure(lm=lm_small)           # set global default
dspy.configure(lm=lm_small, track_usage=True)  # + enable usage tracking

with dspy.context(lm=lm_large):       # temporary override (thread-safe)
    result = module(...)              # uses lm_large inside the block

# ── Modules ───────────────────────────────────────────────────────
pred  = dspy.Predict("q -> a")             # basic call
cot   = dspy.ChainOfThought("q -> a")      # + reasoning field
pot   = dspy.ProgramOfThought("q -> a: float")  # generates + runs code
agent = dspy.ReAct("q -> a", tools=[fn])   # agent loop with tools

# ── Custom Module ─────────────────────────────────────────────────
class MyPipeline(dspy.Module):
    def __init__(self):
        self.step1 = dspy.ChainOfThought("question -> query")
        self.step2 = dspy.ChainOfThought("query, context -> answer")

    def forward(self, question, context=""):
        q = self.step1(question=question).query
        return self.step2(query=q, context=context)

pipe = MyPipeline()
pipe(question="What is X?", context="X is...")

# ── Usage Tracking ────────────────────────────────────────────────
dspy.configure(track_usage=True)
out = pipe(question="...")
print(out.get_lm_usage())   # {'model_name': {'prompt_tokens': N, ...}}

# ── History ───────────────────────────────────────────────────────
gpt_20b.inspect_history(n=2)   # last 2 calls for a specific LM
```